In [ ]:
import os
import glob
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
from google.colab import drive
from PIL import Image
import json
import zipfile
import io
import numpy as np

# Understanding OSI26

In [ ]:
# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Configuration & Path Setup
# Explainability: define the base paths for both annotations and raw images
dataset_name = "OSI26"
base_path = f'/content/drive/My Drive/{dataset_name}'
path_to_labels = f'{base_path}/labels/**/*.xml'
path_to_images = f'{base_path}/images/' # Adjust folder name if necessary

In [ ]:
# Initialize tracking variables for architectural metrics
class_stats = Counter()
total_xml_files = 0
total_size_bytes = 0
resolutions = Counter()

print(f"--- Executing Data Understanding Analysis: {dataset_name} ---")

# 3. Volume Calculation (Comprehensive: Labels + Images)
# Explainability: Summing all files within the dataset folder to report total storage impact
def get_dir_size(path):
    total = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    total += entry.stat().st_size
                elif entry.is_dir():
                    total += get_dir_size(entry.path)
    except Exception:
        pass
    return total

# Calculate total volume of the entire OSI26 directory
total_size_bytes = get_dir_size(base_path)

# 4. Data Extraction Loop (Annotations & Metadata)
# Iterate through each XML file to parse bounding box annotations and image specs
for xml_file in glob.glob(path_to_labels, recursive=True):
    total_xml_files += 1

    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Extract Image Resolution from XML metadata (XAI: Validating source consistency)
        width_node = root.find('.//size/width')
        height_node = root.find('.//size/height')

        if width_node is not None and height_node is not None:
            res_str = f"{width_node.text}x{height_node.text}"
            resolutions[res_str] += 1

        # Count object instances per class
        for obj in root.findall('object'):
            class_name = obj.find('name').text
            class_stats[class_name] += 1

    except Exception as e:
        print(f"[ERROR] Failed to parse {xml_file}: {e}")

# 5. Result Processing for Scientific Documentation
if total_xml_files == 0:
    print("WARNING: No XML files found. Please check your Drive paths (check for 'Labels' vs 'labels').")
else:
    # Convert total bytes to Gigabytes
    total_gb = total_size_bytes / (1024**3)

    # Class Balance Analysis
    df_balance = pd.DataFrame.from_dict(class_stats, orient='index', columns=['Instances']).sort_values(by='Instances', ascending=False)
    total_instances = df_balance['Instances'].sum()
    num_classes = len(df_balance)

    # Final Architectural Summary Table
    print("\n--- DATASET ARCHITECTURAL METRICS (OSI26) ---")
    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames (XML)",
            "Number of Classes",
            "Total Object Instances",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_xml_files,
            num_classes,
            total_instances,
            resolutions.most_common(1)[0][0] if resolutions else "Unknown"
        ]
    }
    df_summary = pd.DataFrame(summary_data)
    print(df_summary.to_string(index=False))

    print("\n--- CLASS DISTRIBUTION & FREQUENCY ---")
    # XAI: Relative frequency helps justify potential model bias or data augmentation needs
    df_balance['Frequency (%)'] = (df_balance['Instances'] / total_instances * 100).round(2)
    print(df_balance)

# Understanding CholecTrack20

In [ ]:
# Locate the first available JSON
json_files = glob.glob('/content/drive/My Drive/datacholec/**/vid*.json', recursive=True)

if json_files:
    with open(json_files[0], 'r') as f:
        data = json.load(f)
        # Get the ID of the first frame that has an annotation
        first_frame_id = list(data['annotations'].keys())[0]
        first_instance = data['annotations'][first_frame_id][0]

        print("--- Structure of an Instance in JSON ---")
        print(json.dumps(first_instance, indent=4))
        print("\n--- Available Category Keys ---")
        print(data['categories'].keys())
else:
    print("No JSON file found. Check the Drive path.")

In [ ]:
# Test the JSON from VID02 that is in the Training subfolder
test_path = '/content/drive/My Drive/datacholec/Training/VID02/vid02.json'

try:
    with open(test_path, 'r') as f:
        sample_data = json.load(f)

    print("--- JSON Structure Discovery ---")
    print(f"Top-level keys: {list(sample_data.keys())}")

    # If there's an 'annotations' key, let's see what it looks like inside
    for key in list(sample_data.keys()):
        if isinstance(sample_data[key], dict):
            print(f"Sub-keys for '{key}': {list(sample_data[key].keys())[:5]} (showing first 5)")
        elif isinstance(sample_data[key], list) and len(sample_data[key]) > 0:
            print(f"Sample item in '{key}': {sample_data[key][0]}")

except Exception as e:
    print(f"Could not open file: {e}")

In [ ]:
import os
import json
import glob
import pandas as pd
from collections import Counter

# 1. Official Mapping (Nwoye et al. 2025)
CHOL_MAP = {
    0: "Grasper", 1: "Bipolar", 2: "Hook", 3: "Scissors", 4: "Clipper",
    5: "Irrigator", 6: "Specimen Bag", 7: "Suction", 8: "Needle Holder",
    9: "Liver Retractor", 10: "Bipolar Scissors"
}

base_path = '/content/drive/My Drive/datacholec'
class_stats = Counter()
challenge_stats = Counter()
resolutions = Counter()
total_instances = 0
total_annotated_frames = 0
total_size_bytes = 0

print("--- Starting Deep Data Understanding: CholecTrack20 ---")

# 2. Volume Calculation (GB)
# Sums all files in the directory (MP4, PNG, JSON, TSV)
for root, dirs, files in os.walk(base_path):
    for f in files:
        total_size_bytes += os.path.getsize(os.path.join(root, f))

# 3. Search for all JSON files
json_files = glob.glob(os.path.join(base_path, '**/vid*.json'), recursive=True)

for j_file in json_files:
    try:
        with open(j_file, 'r') as f:
            data = json.load(f)

        # Extract Resolution from video metadata
        v_info = data.get('video', {})
        res_str = f"{v_info.get('width', '???')}x{v_info.get('height', '???')}"
        resolutions[res_str] += 1

        annotations = data.get('annotations', {})
        for frame_id, instances in annotations.items():
            total_annotated_frames += 1
            for inst in instances:
                c_id = inst.get('instrument')

                if c_id is not None:
                    class_name = CHOL_MAP.get(c_id, f"ID_{c_id}")
                    class_stats[class_name] += 1
                    total_instances += 1

                    # 3. Visual Challenges
                    if inst.get('occluded') == 1: challenge_stats['Occlusion'] += 1
                    if inst.get('bleeding') == 1: challenge_stats['Bleeding'] += 1
                    if inst.get('smoke') == 1: challenge_stats['Smoke'] += 1
                    if inst.get('reflection') == 1: challenge_stats['Reflection'] += 1

    except Exception as e:
        continue

# 4. Results Generation
if total_instances > 0:
    total_gb = total_size_bytes / (1024**3)

    # Standard Metrics Table (Same format as OSI26)
    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames",
            "Number of Classes",
            "Total Object Instances",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_annotated_frames,
            len(class_stats),
            total_instances,
            resolutions.most_common(1)[0][0] if resolutions else "Unknown"
        ]
    }

    print("\n--- DATASET ARCHITECTURAL METRICS (CholecTrack20) ---")
    print(pd.DataFrame(summary_data).to_string(index=False))

    # Class Balance Table
    df_balance = pd.DataFrame.from_dict(class_stats, orient='index', columns=['Total Instances'])
    df_balance = df_balance.sort_values(by='Total Instances', ascending=False)
    df_balance['Frequency (%)'] = (df_balance['Total Instances'] / total_instances * 100).round(2)

    print("\n--- GLOBAL CLASS DISTRIBUTION ---")
    print(df_balance)

    # Visual Challenges Table
    df_challenges = pd.DataFrame.from_dict(challenge_stats, orient='index', columns=['Total Occurrences'])
    df_challenges['Presence (%)'] = (df_challenges['Total Occurrences'] / total_instances * 100).round(2)

    print("\n--- VISUAL CHALLENGES ANALYSIS ---")
    print(df_challenges)
else:
    print("Error: No instances were processed.")

# Understanding ROBUST-MIPS

In [ ]:
zip_path = '/content/drive/My Drive/ROB.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    # List of all JSON files (including subfolders and MACOSX)
    all_jsons = [f for f in z.namelist() if f.endswith('.json')]

    if all_jsons:
        print(f"Total number of JSONs found: {len(all_jsons)}")
        # Let's look at the first JSON that is NOT from MACOSX first.
        real_jsons = [f for f in all_jsons if "_MACOSX" not in f]
        target = real_jsons[0] if real_jsons else all_jsons[0]

        print(f"Inspecting file: {target}")
        with z.open(target) as f:
            sample_data = json.load(f)
            print("\n--- JSON Structure (First Levels) ---")
            print(sample_data.keys() if isinstance(sample_data, dict) else "JSON is a list")
            print("\n--- Content of the first entry ---")
            # Print the first 500 characters so we can understand the pattern
            content_str = json.dumps(sample_data, indent=2)
            print(content_str[:1000])
    else:
        print("No JSON files were found in the ZIP file.")

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/ROB.zip'
dataset_name = "ROBUST-MIPS"

# Metrics
class_stats = Counter() # In ROBUST-MIPS, we count instruments per frame as "classes" or types
resolutions = Counter()
total_instances = 0
total_annotated_frames = 0
total_nodes = 0
total_size_bytes = 0

print(f"--- Starting Deep Data Understanding: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    total_size_bytes = os.path.getsize(zip_path)
    all_files = z.namelist()

    # Filter to ignore Mac metadata
    clean_files = [f for f in all_files if "_MACOSX" not in f and not f.endswith('.DS_Store')]

    for filename in clean_files:
        # A) ANALYZING IMAGES (PNG)
        if filename.lower().endswith('.png'):
            try:
                with z.open(filename) as f:
                    with Image.open(f) as img:
                        resolutions[f"{img.width}x{img.height}"] += 1
            except:
                pass

        # B) ANALYZING ANNOTATIONS (JSON - Tool Poses)
        if filename.lower().endswith('toolposes.json'):
            total_annotated_frames += 1
            try:
                with z.open(filename) as f:
                    data = json.load(f)
                    # data is a list of instruments in the frame
                    num_tools_in_frame = len(data)
                    class_stats[f"{num_tools_in_frame} Tool(s)"] += 1

                    for tool in data:
                        total_instances += 1
                        # Counting nodes (keypoints) to show complexity
                        nodes = tool.get('nodes', [])
                        total_nodes += len([n for n in nodes if n is not None])
            except:
                pass

# 2. Results Generation
if total_instances > 0:
    total_gb = total_size_bytes / (1024**3)

    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames",
            "Total Object Instances",
            "Avg Nodes per Tool",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_annotated_frames,
            total_instances,
            round(total_nodes / total_instances, 2),
            resolutions.most_common(1)[0][0] if resolutions else "Unknown"
        ]
    }

    print(f"\n--- DATASET ARCHITECTURAL METRICS ({dataset_name}) ---")
    print(pd.DataFrame(summary_data).to_string(index=False))

    print("\n--- SCENE COMPLEXITY (Tools per Frame) ---")
    df_balance = pd.DataFrame.from_dict(class_stats, orient='index', columns=['Frames'])
    print(df_balance)
else:
    print("Error: Analysis failed. Verify if JSON files are named correctly.")

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/ROB.zip'
dataset_name = "ROBUST-MIPS"

# Metrics
class_stats = Counter()
resolutions = Counter()
challenge_stats = Counter()
total_instances = 0
total_annotated_frames = 0
total_nodes = 0
total_size_bytes = 0

print(f"--- Starting Deep Data Understanding: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    total_size_bytes = os.path.getsize(zip_path)
    all_files = z.namelist()
    clean_files = [f for f in all_files if "_MACOSX" not in f and not f.endswith('.DS_Store')]

    for filename in clean_files:
        # A) ANALYZING IMAGES (PNG)
        if filename.lower().endswith('.png'):
            try:
                with z.open(filename) as f:
                    with Image.open(f) as img:
                        resolutions[f"{img.width}x{img.height}"] += 1
            except:
                pass

        # B) ANALYZING ANNOTATIONS (JSON)
        if filename.lower().endswith('.json'):
            total_annotated_frames += 1
            try:
                with z.open(filename) as f:
                    data = json.load(f)

                    # If JSON is a list (like the pose skeleton we saw)
                    if isinstance(data, list):
                        num_tools = len(data)
                        class_stats[f"{num_tools} Tool(s)"] += 1
                        for tool in data:
                            total_instances += 1
                            # Check for labels or group_id within the list items
                            label = tool.get('label') or tool.get('group_id') or tool.get('instrument')
                            if label: class_stats[label] += 1

                            # Complexity nodes
                            nodes = tool.get('nodes', [])
                            total_nodes += len([n for n in nodes if n is not None])

                            # Challenges (Missing points act as occlusions)
                            if "missing" in tool.get('tags', []):
                                challenge_stats['Partially Missing/Occluded'] += 1

                    # If JSON is a dict (standard format)
                    elif isinstance(data, dict):
                        # Looking for keys like 'objects' or 'annotations'
                        items = data.get('objects') or data.get('annotations') or []
                        for item in items:
                            total_instances += 1
                            label = item.get('label') or item.get('group_id') or item.get('category_id')
                            if label: class_stats[label] += 1
            except:
                pass

# 2. Results Generation
if total_instances > 0:
    total_gb = total_size_bytes / (1024**3)

    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames",
            "Total Object Instances",
            "Avg Nodes per Tool",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_annotated_frames,
            total_instances,
            round(total_nodes / total_instances, 2) if total_instances > 0 else 0,
            resolutions.most_common(1)[0][0] if resolutions else "Unknown"
        ]
    }

    print(f"\n--- DATASET ARCHITECTURAL METRICS ({dataset_name}) ---")
    print(pd.DataFrame(summary_data).to_string(index=False))

    print("\n--- CLASS & SCENE DISTRIBUTION ---")
    print(pd.Series(class_stats).sort_values(ascending=False))

    if challenge_stats:
        print("\n--- VISUAL CHALLENGES (Keypoint level) ---")
        print(pd.Series(challenge_stats))
else:
    print("Error: No semantic labels found. The dataset might define classes by folder names.")

# Understanding m2cai16-tool-locations

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/mcai.zip'
dataset_name = "m2cai16-tool-locations"

print(f"--- Starting Structural Audit: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = z.namelist()

    # Filtering out Mac metadata and system files
    clean_files = [f for f in all_files if "_MACOSX" not in f and not f.endswith('.DS_Store')]

    # Counting extensions to understand the dataset composition
    ext_counts = {}
    for f in clean_files:
        ext = os.path.splitext(f)[1].lower()
        if ext:
            ext_counts[ext] = ext_counts.get(ext, 0) + 1

    print(f"\nTotal Files (Filtered): {len(clean_files)}")
    print("File Extensions Found:", ext_counts)

    # 2. Searching for potential annotation files (TXT or CSV)
    annotation_samples = [f for f in clean_files if f.endswith('.txt') or f.endswith('.csv')]
    image_samples = [f for f in clean_files if f.endswith('.jpg') or f.endswith('.png')]

    if annotation_samples:
        print(f"\n--- Annotation Inspection ({len(annotation_samples)} files) ---")
        target_ann = annotation_samples[0]
        print(f"Reading sample: {target_ann}")

        with z.open(target_ann) as f:
            # Read first 10 lines to understand the format
            lines = f.readlines()[:10]
            for i, line in enumerate(lines):
                print(f"Line {i}: {line.decode('utf-8').strip()}")
    else:
        print("\n[ALERT] No TXT or CSV files found. Checking if classes are in filenames...")
        # Show first 10 files to see if the structure is "Folder_Name/Image_Name.jpg"
        print("Sample file paths:")
        for f in clean_files[:10]:
            print(f" - {f}")

    if image_samples:
        print(f"\nSample Image Path: {image_samples[0]}")

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/mcai.zip'
dataset_name = "m2cai16-tool-locations"

# Counters
class_stats = Counter()
resolutions = Counter()
total_instances = 0
total_annotated_frames = 0
total_size_bytes = 0

print(f"--- Starting Deep Data Understanding: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    total_size_bytes = os.path.getsize(zip_path)
    all_files = z.namelist()

    # Filter to ignore Mac metadata and focus on XML/JPG
    clean_files = [f for f in all_files if "_MACOSX" not in f and not f.endswith('.DS_Store')]

    for filename in clean_files:
        # A) ANALYZING IMAGES (JPG)
        if filename.lower().endswith('.jpg') and not resolutions:
            try:
                with z.open(filename) as f:
                    with Image.open(f) as img:
                        resolutions[f"{img.width}x{img.height}"] = 1
            except:
                pass

        # B) ANALYZING ANNOTATIONS (XML - Pascal VOC Format)
        if filename.lower().endswith('.xml'):
            total_annotated_frames += 1
            try:
                with z.open(filename) as f:
                    tree = ET.parse(f)
                    root = tree.getroot()

                    # In Pascal VOC, each object is under the <object> tag
                    for obj in root.findall('object'):
                        label = obj.find('name').text
                        class_stats[label] += 1
                        total_instances += 1
            except:
                pass

# 2. Results Generation
if total_instances > 0:
    total_gb = total_size_bytes / (1024**3)

    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames (XML)",
            "Number of Classes",
            "Total Object Instances",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_annotated_frames,
            len(class_stats),
            total_instances,
            list(resolutions.keys())[0] if resolutions else "Unknown"
        ]
    }

    print(f"\n--- DATASET ARCHITECTURAL METRICS ({dataset_name}) ---")
    print(pd.DataFrame(summary_data).to_string(index=False))

    print("\n--- GLOBAL CLASS DISTRIBUTION ---")
    df_balance = pd.DataFrame.from_dict(class_stats, orient='index', columns=['Total Instances'])
    df_balance = df_balance.sort_values(by='Total Instances', ascending=False)
    df_balance['Frequency (%)'] = (df_balance['Total Instances'] / total_instances * 100).round(2)
    print(df_balance)
else:
    print("Error: No instances found in XML files.")

# Understanding Badilla-Solórzano et al. dataset

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/dataset.zip'
dataset_name = "Badilla-Solórzano Analysis"

print(f"--- Starting Structural Audit: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = z.namelist()

    # Filtering out system metadata
    clean_files = [f for f in all_files if "_MACOSX" not in f and not f.endswith('.DS_Store')]

    # Extension Profiling
    ext_counts = {}
    for f in clean_files:
        ext = os.path.splitext(f)[1].lower()
        if ext:
            ext_counts[ext] = ext_counts.get(ext, 0) + 1

    print(f"\nTotal Files (Filtered): {len(clean_files)}")
    print("File Extensions Distribution:", ext_counts)

    # 2. Inspecting the 'masks' zip or folder
    mask_files = [f for f in clean_files if 'mask' in f.lower() and f.endswith('.png')]
    text_files = [f for f in clean_files if f.endswith('.txt')]

    if mask_files:
        print(f"\n--- Mask Inspection (Sample: {mask_files[0]}) ---")
        with z.open(mask_files[0]) as f:
            img = Image.open(f)
            print(f"Mask Mode: {img.mode}") # L = Grayscale, RGB, or P = Paletted
            print(f"Mask Size: {img.size}")
            # Get unique pixel values to see how many classes are in one image
            import numpy as np
            mask_array = np.array(img)
            unique_values = np.unique(mask_array)
            print(f"Unique Pixel Values (Potential Classes): {unique_values}")

    if text_files:
        print(f"\n--- Text Document Inspection (Sample: {text_files[0]}) ---")
        with z.open(text_files[0]) as f:
            content = f.read().decode('utf-8', errors='ignore')[:500]
            print("Content Preview:")
            print(content)

    # 3. Checking for the internal 'masks.zip'
    internal_zips = [f for f in clean_files if f.endswith('.zip') and 'mask' in f.lower()]
    if internal_zips:
        print(f"\n[ALERT] Found internal zip: {internal_zips[0]}")
        print("This may contain the semantic labels. We will need to extract it in the next step.")

In [ ]:
# 1. Configuration
zip_path = '/content/drive/My Drive/dataset.zip'
dataset_name = "Badilla-Solórzano Analysis"

# Counters
class_pixel_stats = Counter() # To count how many frames each class appears in
resolutions = Counter()
total_annotated_frames = 0
total_size_bytes = 0

print(f"--- Starting Deep Data Understanding: {dataset_name} ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    total_size_bytes = os.path.getsize(zip_path)
    all_files = z.namelist()

    # Target only the masks folder (ignoring Mac and metadata)
    mask_files = [f for f in all_files if 'masks/' in f.lower() and f.endswith('.png') and "_MACOSX" not in f]

    for filename in mask_files:
        total_annotated_frames += 1
        try:
            with z.open(filename) as f:
                with Image.open(f) as img:
                    # Record resolution
                    resolutions[f"{img.width}x{img.height}"] += 1

                    # Convert to numpy to find classes
                    mask_array = np.array(img)
                    unique_classes = np.unique(mask_array)

                    # For each class found (ignoring 0/background)
                    for c in unique_classes:
                        if c != 0:
                            class_pixel_stats[c] += 1
        except:
            pass

# 2. Results Generation
if total_annotated_frames > 0:
    total_gb = total_size_bytes / (1024**3)

    # According to the Badilla-Solórzano paper, here is the ID mapping:
    # (Checking the most common IDs in surgical robotics datasets)

    summary_data = {
        "Metric": [
            "Total Dataset Volume",
            "Total Annotated Frames (Masks)",
            "Number of Classes (Detected)",
            "Primary Resolution"
        ],
        "Value": [
            f"{total_gb:.4f} GB",
            total_annotated_frames,
            len(class_pixel_stats),
            resolutions.most_common(1)[0][0] if resolutions else "Unknown"
        ]
    }

    print(f"\n--- DATASET ARCHITECTURAL METRICS (Badilla-Solórzano) ---")
    print(pd.DataFrame(summary_data).to_string(index=False))

    print("\n--- CLASS PRESENCE (Number of Frames per Class ID) ---")
    df_balance = pd.DataFrame.from_dict(class_pixel_stats, orient='index', columns=['Frame Count'])
    df_balance.index.name = 'Class ID'
    print(df_balance.sort_values(by='Frame Count', ascending=False))
else:
    print("Error: No mask files were processed.")

In [ ]:
import zipfile
import numpy as np
from PIL import Image
from scipy.ndimage import label # This identifies separate 'blobs' of pixels
import io

zip_path = '/content/drive/My Drive/dataset.zip'
total_instances = 0
frames_processed = 0

print("--- Counting Physical Instances (Blobs) in Badilla-Solórzano ---")

with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = z.namelist()
    mask_files = [f for f in all_files if 'masks/' in f.lower() and f.endswith('.png') and "_MACOSX" not in f]

    for filename in mask_files:
        try:
            with z.open(filename) as f:
                with Image.open(f) as img:
                    mask_array = np.array(img)

                    # We check each class ID present in the frame (ignoring background 0)
                    unique_classes = np.unique(mask_array)
                    for c in unique_classes:
                        if c == 0: continue

                        # Create a binary mask for JUST this class
                        class_mask = (mask_array == c).astype(int)

                        # Label separate clusters of pixels for this specific class
                        # structure=[[1,1,1],[1,1,1],[1,1,1]] handles diagonal pixels
                        _, num_features = label(class_mask)
                        total_instances += num_features

                    frames_processed += 1
        except:
            pass

print(f"\nResults:")
print(f"- Total Frames Analyzed: {frames_processed}")
print(f"- **Total Object Instances (Actual Count): {total_instances}**")
print(f"- Avg Instruments per Frame: {round(total_instances/frames_processed, 2) if frames_processed > 0 else 0}")